In [36]:
from google.colab import drive
drive.mount('/content/drive')

import cv2
import os
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [37]:
path = "/content/drive/MyDrive/uts_comvis/archive (2)/afhq/train"

In [38]:
data = []
labels = []

limit = 300

for folder in os.listdir(path):
    folder_path = os.path.join(path, folder)

    if not os.path.isdir(folder_path):
        continue

    count = 0

    for file in os.listdir(folder_path):
        if count >= limit:
            break

        img_path = os.path.join(folder_path, file)
        img = cv2.imread(img_path)

        if img is None:
            continue

        img = cv2.resize(img, (64, 64))

        # ===== RGB =====
        mean_rgb = img.mean(axis=(0,1))
        std_rgb = img.std(axis=(0,1))

        # ===== GRAYSCALE =====
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        mean_gray = gray.mean()

        # ===== EDGE =====
        edges = cv2.Canny(gray, 100, 200)
        mean_edge = edges.mean()

        # ===== LAPACIAN (NEW) =====
        laplacian = cv2.Laplacian(gray, cv2.CV_64F)
        mean_lap = laplacian.mean()

        # ===== HISTOGRAM =====
        hist_b = cv2.calcHist([img],[0],None,[16],[0,256]).flatten()
        hist_g = cv2.calcHist([img],[1],None,[16],[0,256]).flatten()
        hist_r = cv2.calcHist([img],[2],None,[16],[0,256]).flatten()

        hist = np.concatenate((hist_b, hist_g, hist_r))
        hist = hist / np.sum(hist)

        # ===== GABUNG FITUR =====
        fitur = np.concatenate((mean_rgb, std_rgb, [mean_gray, mean_edge, mean_lap], hist))

        data.append(fitur)
        labels.append(folder)

        count += 1

data = np.array(data)
labels = np.array(labels)

print("Total data:", len(data))

Total data: 900


In [39]:
scaler = StandardScaler()
data = scaler.fit_transform(data)

In [40]:
X_train, X_test, y_train, y_test = train_test_split(
    data, labels, test_size=0.2, random_state=42
)

In [41]:
model = KNeighborsClassifier(n_neighbors=5)
model.fit(X_train, y_train)

KNeighborsClassifier()

In [42]:
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print("Accuracy:", acc)

Accuracy: 0.5388888888888889


In [43]:
import ipywidgets as widgets
from IPython.display import display, clear_output

upload = widgets.FileUpload(accept='image/*', multiple=False)
button = widgets.Button(description="Prediksi", button_style='success')
output = widgets.Output()

def predict_image(b):
    with output:
        clear_output()

        if len(upload.value) == 0:
            print("Upload gambar dulu!")
            return

        for file_name in upload.value:
            content = upload.value[file_name]['content']
            file_bytes = np.frombuffer(content, np.uint8)
            img = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)

            img = cv2.resize(img, (64, 64))

            # ===== FITUR (HARUS SAMA!) =====
            mean_rgb = img.mean(axis=(0,1))
            std_rgb = img.std(axis=(0,1))

            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            mean_gray = gray.mean()

            edges = cv2.Canny(gray, 100, 200)
            mean_edge = edges.mean()

            # ===== LAPACIAN (NEW) =====
            laplacian = cv2.Laplacian(gray, cv2.CV_64F)
            mean_lap = laplacian.mean()

            hist_b = cv2.calcHist([img],[0],None,[16],[0,256]).flatten()
            hist_g = cv2.calcHist([img],[1],None,[16],[0,256]).flatten()
            hist_r = cv2.calcHist([img],[2],None,[16],[0,256]).flatten()

            hist = np.concatenate((hist_b, hist_g, hist_r))
            hist = hist / np.sum(hist)

            fitur = np.concatenate((mean_rgb, std_rgb, [mean_gray, mean_edge, mean_lap], hist))

            fitur = scaler.transform([fitur])

            hasil = model.predict(fitur)

            plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
            plt.title("Prediksi: " + hasil[0])
            plt.axis("off")
            plt.show()

button.on_click(predict_image)

display(widgets.VBox([
    widgets.HTML("<h2>🐾 Animal Classifier (Improved)</h2>"),
    upload,
    button,
    output
]))